In [ ]:
import subprocess, sys, os

# ── Paths ──────────────────────────────────────────────────────
RDT2_DIR           = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
OUTPUT_DIR         = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-m750-v2"
DATASET_CONFIG_REL = "configs/datasets/custom_dataset.yaml"
LOG_PATH           = "/home/rishabh/Downloads/umi-pipeline-training/train_m750_v2.log"

os.chdir(RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['ACCELERATE_USE_DEEPSPEED']          = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION']  = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']           = 'expandable_segments:True'

cmd = [

    
    sys.executable, 'main.py',
    '--tokenizer_name',               'Qwen/Qwen2.5-VL-7B-Instruct',
    '--vae_name',                      'robotics-diffusion-transformer/RVQActionTokenizer',
    '--pretrained_model_name_or_path', 'robotics-diffusion-transformer/RDT2-VQ',
    '--output_dir',                    OUTPUT_DIR,
    # NO --resume_from_checkpoint → train from scratch (fresh LoRA)
    '--train_batch_size',              '1',
    '--gradient_accumulation_steps',   '8',
    '--eval_batch_size',               '1',
    '--max_train_steps',               '1000',            # ← 1000 steps (not 5000)
    '--eval_strategy',                 'no',
    '--logging_steps',                 '10',
    '--checkpoints_total_limit',       '10',              # ← keep ALL checkpoints
    '--checkpointing_step',            '200',             # ← save every 200 steps
    '--lr_scheduler',                  'cosine',
    '--learning_rate',                 '5e-7',            # ← lower LR (was 1e-6)
    '--mixed_precision',               'bf16',
    '--dataloader_num_workers',        '2',
    '--gradient_checkpointing',
    '--log_level',                     'info',
    '--report_to',                     'none',
    '--lr_warmup_steps',               '50',              # ← warmup (was 0)
    '--dataset',                       DATASET_CONFIG_REL,
    '--image_corruption',
    '--use_default_collate_fn_for_eval',
    '--use_qlora',
    '--lora_r',                        '16',
    '--lora_alpha',                    '16',
    '--lora_dropout',                  '0.1',
]

print("🚀 Training RDT2-VQ for M750 (FRESH — fixed settings)")
print(f"   Output:    {OUTPUT_DIR}")
print(f"   LR:        5e-7 (was 1e-6)")
print(f"   Steps:     1000 (was 5000)")
print(f"   Warmup:    50 steps (was 0)")
print(f"   Save every: 200 steps → checkpoints at 200, 400, 600, 800, 1000")
print(f"   Log:       {LOG_PATH}")
print()
print("   ⚠️  Watch the loss! If it hits 0.0 before step 500, stop training")
print("       and use the last checkpoint where loss > 0.01")
print("=" * 70)

with open(LOG_PATH, 'w') as logf:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy()
    )
    for line in process.stdout:
        logf.write(line)
        logf.flush()
        print(line, end='', flush=True)

process.wait()
print("\n✅ Done!" if process.returncode == 0 else f"\n❌ Exit code {process.returncode}")

🚀 Training RDT2-VQ for M750 (FRESH — fixed settings)
   Output:    /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-m750-v2
   LR:        5e-7 (was 1e-6)
   Steps:     1000 (was 5000)
   Warmup:    50 steps (was 0)
   Save every: 200 steps → checkpoints at 200, 400, 600, 800, 1000
   Log:       /home/rishabh/Downloads/umi-pipeline-training/train_m750_v2.log

   ⚠️  Watch the loss! If it hits 0.0 before step 500, stop training
       and use the last checkpoint where loss > 0.01

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.73s/it]
Trainable parameters: 49239808
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. N

In [1]:
"""
STEP 2: Train VQ-VAE Tokenizer for D=7 (M750 + UMI gripper)
=============================================================
Run from RDT2 directory:
    cd /home/rishabh/Downloads/umi-pipeline-training/RDT2
    source /home/rishabh/Downloads/umi-pipeline-training/umi_env/bin/activate
    python step2_train_vqvae.py
"""

import os, sys, glob, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── CONFIG ────────────────────────────────────────────────────────────────────
SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"
OUTPUT_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-7dof"
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
ACTION_DIM   = 7        # [x, y, z, rx, ry, rz, gripper]
ACTION_HZ    = 24       # action chunk size
BATCH_SIZE   = 32
EPOCHS       = 100
LR           = 1e-4
DEVICE       = "cuda:0" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Shards: {SHARDS_DIR}")
print(f"Output: {OUTPUT_DIR}")

# ── PATCH multivqvae.py FOR D=7 ───────────────────────────────────────────────
print("\n[1/4] Patching multivqvae.py for D=7...")

VQVAE_FILE = f"{RDT2_DIR}/vqvae/models/multivqvae.py"

# Backup original
import shutil
if not os.path.exists(VQVAE_FILE + ".bak_original"):
    shutil.copy(VQVAE_FILE, VQVAE_FILE + ".bak_original")
    print(f"  Backed up to {VQVAE_FILE}.bak_original")

D7_CODE = '''"""
multivqvae.py — PATCHED FOR M750 SINGLE ARM D=7
D=7 layout: [x, y, z, rx, ry, rz, gripper]
              0  1  2   3   4   5      6
"""
from typing import Union
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from vqvae.models.vqvae import VQVAE


def select_act_dim(x, act_type):
    """Extract sub-dims from D=7 tensor."""
    if act_type == "pos":   return x[..., 0:3]   # xyz
    elif act_type == "rot": return x[..., 3:6]   # rx,ry,rz
    elif act_type == "grip":return x[..., 6:7]   # gripper
    raise ValueError(f"Unknown: {act_type}")


def apply_act_dim(x, y, act_type):
    """Write sub-dims back into D=7 tensor."""
    if act_type == "pos":   x[..., 0:3] = y
    elif act_type == "rot": x[..., 3:6] = y
    elif act_type == "grip":x[..., 6:7] = y
    return x


class MultiVQVAE(nn.Module, PyTorchModelHubMixin):
    def __init__(self, input_dim, embedding_dim, cnn_config,
                 num_embeddings, action_horizon,
                 n_codebooks=[6,2,1],
                 codebook_restart_interval=64,
                 commitment_cost=0.25,
                 codebook_cost=0., local_rank=0):
        super().__init__()

        self.pos_vqvae = VQVAE(
            input_dim=input_dim["pos"],        # 3
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["pos"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.pos_id_len = n_codebooks["pos"] * 3

        self.rot_vqvae = VQVAE(
            input_dim=input_dim["rot"],        # 3
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["rot"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.rot_id_len = n_codebooks["rot"] * 3

        self.grip_vqvae = VQVAE(
            input_dim=input_dim["grip"],       # 1
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["grip"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.grip_id_len = n_codebooks["grip"] * 3

        self.action_dim     = input_dim["pos"] + input_dim["rot"] + input_dim["grip"]  # = 7
        self.action_horizon = action_horizon
        self.num_embeddings = num_embeddings

    def encode(self, x):
        if isinstance(x, dict):
            p, r, g = x["pos"], x["rot"], x["grip"]
        else:
            p = select_act_dim(x, "pos")
            r = select_act_dim(x, "rot")
            g = select_act_dim(x, "grip")
        return torch.cat([
            self.pos_vqvae.encode(p),
            self.rot_vqvae.encode(r),
            self.grip_vqvae.encode(g)], dim=-1)

    def decode(self, ids, return_dict=False):
        p_ids = ids[..., :self.pos_id_len]
        r_ids = ids[..., self.pos_id_len:self.pos_id_len+self.rot_id_len]
        g_ids = ids[..., self.pos_id_len+self.rot_id_len:]
        p = self.pos_vqvae.decode(p_ids)
        r = self.rot_vqvae.decode(r_ids)
        g = self.grip_vqvae.decode(g_ids)
        if return_dict:
            return {"pos": p, "rot": r, "grip": g}
        x = torch.zeros(ids.shape[0], self.action_horizon,
                        self.action_dim, device=ids.device)
        x = apply_act_dim(x, p, "pos")
        x = apply_act_dim(x, r, "rot")
        x = apply_act_dim(x, g, "grip")
        return x   # (B, T, 7)

    def calculate_loss(self, x, x_recon):
        return {
            "pos":  self.pos_vqvae.calculate_loss(
                        select_act_dim(x,"pos"),
                        select_act_dim(x_recon,"pos"), act_type="pos"),
            "rot":  self.rot_vqvae.calculate_loss(
                        select_act_dim(x,"rot"),
                        select_act_dim(x_recon,"rot"), act_type="rot"),
            "grip": self.grip_vqvae.calculate_loss(
                        select_act_dim(x,"grip"),
                        select_act_dim(x_recon,"grip"), act_type="grip"),
        }
'''

with open(VQVAE_FILE, "w") as f:
    f.write(D7_CODE)
print("  ✅ multivqvae.py patched for D=7")

# ── DATASET ───────────────────────────────────────────────────────────────────
print("\n[2/4] Loading M750 shard dataset...")

class M750Dataset(Dataset):
    def __init__(self, shards_dir, action_horizon=24):
        self.chunks = []
        files = sorted(glob.glob(f"{shards_dir}/**/*.pt", recursive=True))
        if not files:
            files = sorted(glob.glob(f"{shards_dir}/*.pt"))
        print(f"  Found {len(files)} shard files")

        for f in files:
            try:
                data = torch.load(f, map_location="cpu")
                # Handle different shard formats
                if isinstance(data, dict):
                    if "action" in data:
                        actions = data["action"]
                    elif "actions" in data:
                        actions = data["actions"]
                    else:
                        print(f"  Skip {f}: no 'action' key, keys={list(data.keys())}")
                        continue
                elif isinstance(data, torch.Tensor):
                    actions = data
                else:
                    continue

                actions = actions.float()
                if actions.ndim == 1:
                    actions = actions.unsqueeze(0)

                # Verify D=7
                if actions.shape[-1] != 7:
                    print(f"  Skip {f}: action dim={actions.shape[-1]} (need 7)")
                    continue

                # Sliding window chunks
                T = len(actions)
                stride = action_horizon // 2
                for i in range(0, T - action_horizon + 1, stride):
                    chunk = actions[i:i+action_horizon]
                    if len(chunk) == action_horizon:
                        self.chunks.append(chunk)
            except Exception as e:
                print(f"  Skip {f}: {e}")

        print(f"  Total chunks: {len(self.chunks)}")
        if len(self.chunks) == 0:
            raise RuntimeError(f"No D=7 data found in {shards_dir}!")

        # Print action stats
        all_actions = torch.stack(self.chunks).reshape(-1, 7)
        print(f"  Action min: {all_actions.min(0).values.tolist()}")
        print(f"  Action max: {all_actions.max(0).values.tolist()}")
        print(f"  Action mean:{all_actions.mean(0).tolist()}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]  # (T, 7)

dataset = M750Dataset(SHARDS_DIR, action_horizon=ACTION_HZ)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                     shuffle=True, num_workers=2, pin_memory=True)

# ── BUILD VQVAE ───────────────────────────────────────────────────────────────
print("\n[3/4] Building VQ-VAE for D=7...")

from vqvae.models.multivqvae import MultiVQVAE

VQVAE_CONFIG = {
    "input_dim":    {"pos": 3, "rot": 3, "grip": 1},
    "embedding_dim": 64,
    "cnn_config": {
        "channels": [64, 128, 256],
        "kernels":  [4, 4, 4],
        "strides":  [2, 2, 2],
    },
    "num_embeddings":  512,
    "action_horizon":  ACTION_HZ,
    "n_codebooks":  {"pos": 6, "rot": 2, "grip": 1},
}

vqvae = MultiVQVAE(**VQVAE_CONFIG).to(DEVICE)
print(f"  action_dim     = {vqvae.action_dim}")    # should be 7
print(f"  pos_id_len     = {vqvae.pos_id_len}")    # 18
print(f"  rot_id_len     = {vqvae.rot_id_len}")    # 6
print(f"  grip_id_len    = {vqvae.grip_id_len}")   # 3
print(f"  valid_id_len   = {vqvae.pos_id_len + vqvae.rot_id_len + vqvae.grip_id_len}")  # 27

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print(f"\n[4/4] Training VQ-VAE for {EPOCHS} epochs...")

optimizer = torch.optim.Adam(vqvae.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5)

best_loss = float("inf")

for epoch in range(EPOCHS):
    vqvae.train()
    total = {"pos": 0., "rot": 0., "grip": 0., "total": 0.}
    n = 0

    for batch in loader:
        batch = batch.to(DEVICE)          # (B, T, 7)
        ids   = vqvae.encode(batch)       # (B, L)
        recon = vqvae.decode(ids)         # (B, T, 7)
        losses = vqvae.calculate_loss(batch, recon)

        loss = sum(v["loss"] for v in losses.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vqvae.parameters(), 1.0)
        optimizer.step()

        total["total"] += loss.item()
        for k in ["pos", "rot", "grip"]:
            total[k] += losses[k]["loss"].item()
        n += 1

    scheduler.step()
    avg = {k: v/n for k, v in total.items()}

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS} | "
              f"total={avg['total']:.4f} | "
              f"pos={avg['pos']:.4f} | "
              f"rot={avg['rot']:.4f} | "
              f"grip={avg['grip']:.4f}")

    # Save checkpoint every 25 epochs
    if (epoch + 1) % 25 == 0:
        ckpt = f"{OUTPUT_DIR}/vqvae_epoch{epoch+1}.pt"
        torch.save({
            "epoch": epoch+1,
            "model": vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss": avg["total"],
        }, ckpt)
        print(f"  💾 Saved: {ckpt}")

    # Save best
    if avg["total"] < best_loss:
        best_loss = avg["total"]
        torch.save({
            "epoch": epoch+1,
            "model": vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss": best_loss,
        }, f"{OUTPUT_DIR}/vqvae_best.pt")

# Final save
torch.save({
    "epoch": EPOCHS,
    "model": vqvae.state_dict(),
    "config": VQVAE_CONFIG,
    "loss": avg["total"],
}, f"{OUTPUT_DIR}/vqvae_final.pt")

print(f"\n✅ VQ-VAE training complete!")
print(f"   Best loss : {best_loss:.4f}")
print(f"   Saved to  : {OUTPUT_DIR}/vqvae_final.pt")
print(f"\nNext: Run step3_build_normalizer.py")

Device: cuda:0
Shards: /home/rishabh/Downloads/umi-pipeline-training/shards
Output: /home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-7dof

[1/4] Patching multivqvae.py for D=7...
  Backed up to /home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/multivqvae.py.bak_original
  ✅ multivqvae.py patched for D=7

[2/4] Loading M750 shard dataset...
  Found 0 shard files
  Total chunks: 0


RuntimeError: No D=7 data found in /home/rishabh/Downloads/umi-pipeline-training/shards!